# Замыкания и декораторы

**Результат:** Создавать функции с сохранёнными параметрами и обёртки с прежним контрактом.


## 1. Замыкание

Замыкание — функция, использующая переменные окружающей области после завершения внешнего вызова. Так можно настроить правило один раз и передавать готовую функцию как зависимость.

Вложенная функция обращается к переменным внешней функции. Каждый вызов фабрики создаёт своё окружение. Если нужно переприсвоить захваченную переменную, используется nonlocal; без него присваивание создаст локальную переменную. Изменять объект по ссылке и переприсваивать имя — разные действия.


In [36]:
def make_prefix(prefix):
    def format_text(text):
        return prefix + text
    return format_text

api = make_prefix("Привет, ")
assert api("ok") == "API: ok"
assert make_prefix("SQL: ")("ok") == "SQL: ok"


AssertionError: 

In [37]:
print(api("Вася"))
print(api("Коля"))
print(api("Гриша"))

Привет, Вася
Привет, Коля
Привет, Гриша


In [3]:
prefix = "Привет, "

def add_prefix(prefix):
    print(123)
    def add_text(text):
        return prefix + text # 
    return add_text

add_text = add_prefix("API: ")
result = add_text("SQL")
result

123


'API: SQL'

In [26]:
def sql_query_select():
    select_template = "SELECT * FROM {} WHERE {}"
    def make_sql_query_with_data(table, *filters):

        conditions = " AND ".join(filters)

        result_sql_query = select_template.format(table, conditions)
        print(result_sql_query)
    return make_sql_query_with_data

make_sql = sql_query_select()

In [27]:
make_sql("users", "age > 18", "age < 30") 

SELECT * FROM users WHERE age > 18 AND age < 30


In [28]:
make_sql("clients", "age > 21") 

SELECT * FROM clients WHERE age > 21


In [8]:
name, clients = "name", "clients, drop table users"
"SELECT {}, city FROM {};".format(name, clients)

'SELECT name, city FROM clients, drop table users;'

In [ ]:
import sqlite3

# Подключаемся к базе (для примера берем временную базу в памяти)
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Создаем таблицу и добавляем тестовые данные
cursor.execute("CREATE TABLE users (name TEXT, age INTEGER)")
cursor.execute("INSERT INTO users VALUES ('Иван', 25), ('Мария', 30)")

# --- ПРИМЕР ПАРАМЕТРИЗАЦИИ ---

# 1. Данные от пользователя (даже если он пытается нас взломать)
user_name = "Иван' OR '1'='1"
age = 18

# 2. Пишем запрос с заглушкой '?'
sql_query = "SELECT * FROM users WHERE name = ? or age > ?"

# 3. Передаем данные вторым аргументом в виде КОРТЕЖА (обрати внимание на запятую!)
cursor.execute(sql_query, (user_name, age))

# Проверяем результат
result = cursor.fetchall()
print(result) 
# Вывод: [] (пустой список)
# База искала человека с именем "Иван' OR '1'='1", не нашла его и ничего не вернула.
# Твои данные остались в безопасности!

In [39]:
def filter_condition(item):
    return item >= 5

numbers = [1,5,6,2]

print(list(filter(filter_condition, numbers)))

[5, 6]


In [ ]:
def create_threshold_filter(threshold):
    # Внешняя функция запоминает 'threshold'
    def filter_func(item):
        return item > threshold  # Внутренняя функция использует 'threshold'
    return filter_func

numbers = [10, 25, 40, 5, 60]

# Создаем два разных фильтра "на лету"
filter_above_20 = create_threshold_filter(20)
filter_above_50 = create_threshold_filter(50)

# Используем их в filter()
print(list(filter(filter_above_20, numbers)))  # [25, 40, 60]
print(list(filter(filter_above_50, numbers)))  # [60]

**Практика:** Создать два независимых форматтера с разными префиксами.


## 2. Обёртка и @

Декоратор принимает функцию и возвращает заменяющий вызываемый объект. Обычно это обёртка, добавляющая проверку, учёт или подготовку вокруг исходного вызова. Запись @decorate перед def эквивалентна последующему присваиванию f = decorate(f).

Обёртка должна передать *args и **kwargs и вернуть результат исходной функции. Если забыть return, вызывающий получит None. Если без причины перехватить исключение, изменится контракт. wraps переносит имя, docstring и ссылку __wrapped__ на исходную функцию.


In [ ]:
from functools import wraps
def stripped_result(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs).strip()
    return wrapper

@stripped_result
def greeting(name):
    """Вернуть приветствие."""
    return f" hello {name} "

assert greeting(name="ira") == "hello ira"
assert greeting.__name__ == "greeting"


**Практика:** Объяснить разницу между @stripped_result и вызовом greeting.


## 3. Декоратор с настройкой

Для настройки нужны два этапа: фабрика принимает параметр, возвращает декоратор, а декоратор получает функцию. Поэтому @add_prefix("! ") сначала создаёт настроенный декоратор.

Не стоит декорировать всё подряд: явный вызов простой функции может быть понятнее. При создании замыканий в цикле переменная захватывается как переменная, а не снимок её значения; отдельная фабрика даёт независимую настройку для каждого результата.


In [ ]:
def add_prefix(prefix):
    def decorate(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            return prefix + function(*args, **kwargs)
        return wrapper
    return decorate

@add_prefix("! ")
def label(text):
    return text.upper()

assert label("api") == "! API"


**Практика:** Назвать значения, принимаемые каждым из трёх уровней функций.


Практические задания находятся в [homework.ipynb](homework.ipynb).


## Справочные источники

- [functools.wraps](https://docs.python.org/3.14/library/functools.html#functools.wraps)
